# Rapprochement automatique paiements / factures — pipeline

Chaque étape du brief est une méthode de `Project` (`src/api.py`). Les résultats sont écrits sur disque
(`data/interim`, `reports/`, `models/`) et renvoyés sous forme de tableaux.

- **Paramètres** : `config/settings.yaml` (par étape) et `config/rules.yaml` (règles de l'étape 4),
  éditables à la main, depuis l'interface (`streamlit run src/ui/app.py`) ou depuis ce notebook.
- **Données réelles** : renseigner `config/schema.yaml`, puis `Project(dataset="real")`.
- **Données synthétiques** : `Project(dataset="synthetic")` — volume réglable (réel : ~2 M de paiements ;
  commencer par 50 000 pour une exécution de quelques minutes).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import altair as alt
import pandas as pd
from IPython.display import Markdown, display

from src.api import Project

pd.set_option("display.max_columns", 50)
project = Project(dataset="synthetic", log=print)
N_PAYMENTS = 50_000          # 2_000_000 pour le volume réel (≈ 1 h de bout en bout)

## Paramètres

Lecture et modification des paramètres depuis le notebook (écrits dans `config/settings.yaml`).

In [ ]:
from src.settings import save_settings

settings = project.settings
print(settings.split)
print(settings.allocation.signals)
# Exemple : settings.split.purge_days = 7 ; save_settings(settings, project.settings_path)

## Étape 1 — chargement, normalisation, journal

In [ ]:
step1 = project.load(n_payments=N_PAYMENTS)
display(step1["volumes"])
display(step1["issues"][step1["issues"]["count"] > 0])
display(step1["missing_fields"][~step1["missing_fields"]["mapped"] | (step1["missing_fields"]["null"] > 0)])
step1["meta"]

## Étape 2 — découpage temporel et boucle quotidienne

In [ ]:
split = project.split()
display(pd.DataFrame([{"période": p.name, "début": p.start, "fin": p.end, "jours": p.days} for p in split.periods]))

replay = project.replay("test", "null")        # rapprocheur vide : taille des lots et reliquat
daily = replay["daily"].melt(id_vars="day", value_vars=["new", "carried"], var_name="origine", value_name="paiements")
alt.Chart(daily).mark_bar().encode(x="day:T", y="paiements:Q", color="origine:N").properties(height=240)

## Étape 3 — allocation (rappel cible ≥ 99 %)

In [ ]:
alloc = project.measure_allocation("validation")
display(pd.Series(alloc["summary"]))
display(alloc["by_route"])
display(alloc["found_by"].head(10))

## Étape 4 — réconciliation par règles (baseline)

In [ ]:
rules = project.backtest("test", matcher="rules")
display(pd.Series(rules["summary"]))
display(rules["by_rule_alone"])
display(rules["by_rule"])

## Étape 5 — réconciliation ML (entraînement sur train, seuils sur la validation)

`project.train()` rejoue entraînement + validation, entraîne le modèle et calibre les seuils sur la boucle
réelle de validation. Après un changement des paramètres de décision (marge δ, précision cible),
`project.calibrate()` recalcule seulement les seuils, sans réentraîner.

In [ ]:
meta = project.train()
display(pd.Series(meta["metrics"]["validation"]))
display(pd.Series(meta["thresholds"]["kinds"], name="τ_high par type"))
pd.read_csv(project.model_dir / "feature_importance.csv").head(15)

## Étape 6 — backtest complet et tableau en cascade

In [ ]:
ev = project.backtest("test", matcher="pipeline")
display(pd.Series(ev["summary"]))
display(ev["cascade"])
display(ev["by_group"])
display(ev["by_step"])

In [ ]:
curve = ev["curve"].replace([float("inf")], None).dropna()
target = ev["summary"]["précision_cible"]
line = alt.Chart(curve).mark_line().encode(x=alt.X("taux_automatisation:Q", axis=alt.Axis(format="%")),
                                           y=alt.Y("précision:Q", scale=alt.Scale(zero=False), axis=alt.Axis(format="%")))
rule = alt.Chart(pd.DataFrame({"y": [target]})).mark_rule(strokeDash=[4, 4]).encode(y="y:Q")
(line + rule).properties(height=260, title="Automatisation / précision (règles acquises, seuil ML variable)")

In [ ]:
for name in ("by_month", "by_client_file", "ml_calibration"):
    if name in ev:
        display(Markdown(f"**{name}**"))
        display(ev[name])
display(Markdown((project.reports_dir / "evaluation_pipeline_test.md").read_text(encoding="utf-8")))